# Llibreries

In [53]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import os
import sys
from pathlib import Path

# Visualització
import matplotlib.pyplot as plt
import seaborn as sns

# Config vis
sns.set_theme()

# Funcions
cwd = os.getcwd()
parent = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.insert(0, parent)
from src.utils import neteja_noms_columnes

# Dimensions
Carrega del dataset que conté totes les dimensions de les dades

In [2]:
dimensions = pd.read_csv("../data/dimensions/pad_dimensions.csv")
dim_barris = pd.read_csv("../data/dimensions/BarcelonaCiutat_Barris.csv")

## Dades Urbanes
En aquest apartat s'ingestaran i transformaran les dades de caracter urbà. 

### Incidents Urbans

En alguns articles, s' introdueixen variables de criminalitat per estudiar la gentrificació. Si bé no s' ha pogut aconseguir índexs de criminalitats ja creats, es pot agregar informació sobre incidents a nivell de barri. 

#### 2025

In [4]:
df_incidents_25_raw = pd.read_csv("../data/raw/urbanes/2025_incidents_gestionats_gub.csv")
df_incidents_25_raw.head()

,Codi_Incident,Descripcio_Incident,Codi_districte,Nom_districte,Codi_barri,Nom_barri,NK_Any,Mes_any,Nom_mes,Numero_incidents_GUB
0,000,INCENDIS,NaN,NaN,NaN,NaN,2025,1,gener,2
1,000,INCENDIS,NaN,NaN,NaN,NaN,2025,2,febrer,8
2,000,INCENDIS,NaN,NaN,NaN,NaN,2025,3,març,1
3,000,INCENDIS,NaN,NaN,NaN,NaN,2025,4,abril,4
4,000,INCENDIS,NaN,NaN,NaN,NaN,2025,5,maig,3


In [5]:
df_incidents_25_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29999 entries, 0 to 29998
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Codi_Incident         29999 non-null  object 
 1   Descripcio_Incident   29999 non-null  object 
 2   Codi_districte        29640 non-null  float64
 3   Nom_districte         29640 non-null  object 
 4   Codi_barri            29640 non-null  float64
 5   Nom_barri             29640 non-null  object 
 6   NK_Any                29999 non-null  int64  
 7   Mes_any               29999 non-null  int64  
 8   Nom_mes               29999 non-null  object 
 9   Numero_incidents_GUB  29999 non-null  int64  
dtypes: float64(2), int64(3), object(5)
memory usage: 2.3+ MB


**Observacions:**
- Tal i com es pot observar, en aquest dataset existeixen valors nuls. 
- Els formats de les dades semblen adequats.


Els nuls semblen estar acumulats en els atributs amb informació sobre districte i barri. Donada la importància de no tenir aquesta dada buida, començarem eliminant les files on almenys codi_barri és NaN.

In [16]:
df_incidents_25_mod  = df_incidents_25_raw.dropna(subset=["Codi_barri"])
df_incidents_25_mod.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29640 entries, 11 to 29998
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Codi_Incident         29640 non-null  object 
 1   Descripcio_Incident   29640 non-null  object 
 2   Codi_districte        29640 non-null  float64
 3   Nom_districte         29640 non-null  object 
 4   Codi_barri            29640 non-null  float64
 5   Nom_barri             29640 non-null  object 
 6   NK_Any                29640 non-null  int64  
 7   Mes_any               29640 non-null  int64  
 8   Nom_mes               29640 non-null  object 
 9   Numero_incidents_GUB  29640 non-null  int64  
dtypes: float64(2), int64(3), object(5)
memory usage: 2.5+ MB


**Observacions:**
- Es pot observar com el nombre de registres ha disminuit, i ara no s' hi detecta cap valor nul en les dades. 
- El codi barri és un valor categòric i per tant hauria d'estar com int32 almenys.

In [22]:
df_incidents_25_mod.loc[:, "Codi_barri"] = df_incidents_25_mod["Codi_barri"].astype("int32")

In [51]:
# Agrupem les dades per tipologia d'incident i barri
df_incidents_25_grouped = df_incidents_25_mod.groupby(["Codi_barri"]).size().reset_index(name = 'total_incidents')
df_incidents_25_grouped.head()

,Codi_barri,total_incidents
0,1,561
1,2,508
2,3,491
3,4,498
4,5,479


#### 2015

In [37]:
df_incidents_15_raw = pd.read_csv("../data/raw/urbanes/2015_incidents_gestionats_gub.csv")
df_incidents_15_raw.head()

,Codi Incident,Descripció Incident,Codi districte,Nom districte,Codi barri,Nom barri,NK Any,Mes de any,Nom mes,Número d'incidents GUB
0,222,INFRACCIONS EN MOVIMENT ...,6,Gràcia,32,el Camp d'en Grassot i Gràcia Nova,2015,5,Maig,10
1,400,CONVIVÈNCIA VEINAL ...,10,Sant Martí,69,Diagonal Mar i el Front Marítim del Poblenou,2015,5,Maig,13
2,420,INCIDÈNCIES DE LOCALS ...,2,Eixample,7,la Dreta de l'Eixample,2015,5,Maig,43
3,660,VIOLÈNCIA DOMÈSTICA ...,9,Sant Andreu,61,la Sagrera,2015,4,Abril,1
4,203,SENYALS AUTOMATITZADES ...,2,Eixample,5,el Fort Pienc,2015,4,Abril,19


In [38]:
df_incidents_15_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33801 entries, 0 to 33800
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Codi Incident           33801 non-null  object
 1   Descripció Incident     33801 non-null  object
 2   Codi districte          33801 non-null  int64 
 3   Nom districte           33801 non-null  object
 4   Codi barri              33801 non-null  int64 
 5   Nom barri               33801 non-null  object
 6   NK Any                  33801 non-null  int64 
 7   Mes de any              33801 non-null  int64 
 8   Nom mes                 33801 non-null  object
 9   Número d'incidents GUB  33801 non-null  int64 
dtypes: int64(5), object(5)
memory usage: 2.6+ MB


**Observacions:**
- No es detecten valors nuls per al conjunt d' incidents del 2015.
- Codi Barri correctament importada com int.

In [50]:
df_incidents_15_grouped = df_incidents_15_raw.groupby(["Codi barri"]).size().reset_index(name = 'total_incidents')
df_incidents_15_grouped.head()

,Codi barri,total_incidents
0,-1,564
1,1,674
2,2,625
3,3,615
4,4,592


In [48]:
df_incidents_15_grouped["Codi barri"].unique()

array([-1,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73], dtype=int64)

**Observacions:**
- S'observa un codi de barri marcat com -1. No existeix a la nostra dimensió de barris,i per tant pot referir-se al localitzacions desconegudes.

### Comerços
En aquest apartat, integrem dades dels comerços als barris.

In [55]:
with ZipFile("../data/Taula estadística - Nombre de locals comercials actius per sector d’activitat i grup d’activitat.zip", 'r') as zip_ref:
    zip_ref.extractall("../data/raw/urbanes/locals_comercials")

In [60]:
df_locals = pd.read_csv("../data/raw/urbanes/locals_comercials/Taula estadística.csv")
df_locals.head()

,Territori,Tipus de territori,Sector activitat,31 Des. 2016,31 Des. 2016.1,31 Des. 2016.2,31 Des. 2016.3,31 Des. 2016.4,31 Des. 2016.5,31 Des. 2016.6,...,31 Des. 2024.5,31 Des. 2024.6,31 Des. 2024.7,31 Des. 2024.8,31 Des. 2024.9,31 Des. 2024.10,31 Des. 2024.11,31 Des. 2024.12,31 Des. 2024.13,31 Des. 2024.14
0,Territori,Tipus de territori,Sector activitat,Quotidià alimentari,Quotidià no alimentari,Parament de la llar,Equipament personal,Oci i cultura,Automoció,Activitats immobiliàries,...,Automoció,Activitats immobiliàries,Ensenyament,Equipaments culturals i recreatius,Finances i assegurances,"Manteniment, neteja i producció",Reparacions (Electrodomèstics i automòbils),"Restaurants, bars i hotels (Inclòs hostals, pe...",Sanitat i assistència,Altres
1,el Raval,Barri,Comerç al detall,561,71,88,208,124,1,-,...,2,-,-,-,-,-,-,-,-,102
2,el Raval,Barri,Serveis,-,-,-,-,-,-,12,...,-,6,49,55,27,-,10,537,14,428
3,el Raval,Barri,Altres,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,220
4,el Barri Gòtic,Barri,Comerç al detall,137,57,49,484,52,-,-,...,-,-,-,-,-,-,-,-,-,216


**Observacions.**
- El format és diferent ja que en aquest cas les dades provenen d' una taula estadística i no de Microdades. 
- Per tal de poder treballar de manera correcte, haurem de convertir el format a llarg en comptes de *wide*.
- Es pot observar també que no existeixen dades del 2015 ni 2023. Utilitzarem el conjunt de 31/12/2014 com a aproximació per a les dades de 2015 i les dades de 2022 per a l'aproximació de les dades de 2023.

In [ ]:
df_locals_melt = df_locals.melt(id_vars=["Territori", "Tipus de territori"], var_name="activitat", value_name="Nombre_locals")
df_locals_melt.head()

,Territori,Tipus de territori,Sector activitat,Nombre_locals
0,Territori,Tipus de territori,Sector activitat,Sector activitat
1,el Raval,Barri,Sector activitat,Comerç al detall
2,el Raval,Barri,Sector activitat,Serveis
3,el Raval,Barri,Sector activitat,Altres
4,el Barri Gòtic,Barri,Sector activitat,Comerç al detall
